In [ ]:
# Imports and the grid point walked through: the headline configuration's market, window length and construction from config (run scripts/data_processing.py first).
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from regime_networks import config
from regime_networks.networks import load_networks
from regime_networks.graph2vec import Graph2Vec, _wl_graph_features
from regime_networks.reduction import fit_reduction, transform_reduction
from regime_networks.regimes import fit_and_rank_regimes, empirical_transitions
from regime_networks.validation import split_train_test, load_vix, validate_regime
from regime_networks.plotting import set_publication_style, plot_regime_timeline, plot_vix_overlay, plot_transition_comparison

INDEX_CODE, WINDOW_LENGTH, GRAPH_TYPE = config.PRIMARY_INDEX, config.HEADLINE_WINDOW[0], config.HEADLINE_GRAPH_TYPE
set_publication_style()

In [ ]:
# Load the rolling-window networks and split windows at the train/test cutoff by their end date.
data = load_networks(INDEX_CODE, WINDOW_LENGTH)
train, test = split_train_test(data)
train_graphs, test_graphs = train[config.GRAPH_KEYS[GRAPH_TYPE]], test[config.GRAPH_KEYS[GRAPH_TYPE]]
print(f"{len(data['stock_names'])} stocks, {len(train_graphs)} train windows (to {train['window_dates'][-1].date()}), {len(test_graphs)} test windows (to {test['window_dates'][-1].date()})")

In [ ]:
# What Graph2Vec reads from one window: a bag of Weisfeiler-Lehman labels, starting from node degrees and adding one round of hashed neighbourhood labels per iteration.
G = train_graphs[0]
features = _wl_graph_features(G, config.G2V_WL_ITERATIONS)
print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges -> {len(features)} WL labels, {len(set(features))} distinct")
print("most common starting (degree) labels:", Counter(features[:G.number_of_nodes()]).most_common(5))

In [ ]:
# Learn one embedding per train window (Doc2Vec PV-DBOW, one document per graph), then infer test windows against the frozen model without refitting.
g2v = Graph2Vec()
train_raw, train_oov = g2v.fit(train_graphs)
test_raw, test_oov = g2v.infer(test_graphs)
print(f"embeddings: train {train_raw.shape}, test {test_raw.shape}; mean share of labels outside the train vocabulary: train {np.mean(train_oov):.3f}, test {np.mean(test_oov):.3f}")

In [ ]:
# Standardise and keep the principal components explaining 95% of variance, both fit on train windows only.
X_train, scaler, pca = fit_reduction(train_raw)
X_test = transform_reduction(test_raw, scaler, pca)
print(f"{train_raw.shape[1]} dimensions -> {pca.n_components_} principal components")

In [ ]:
# Cluster with K-means and a Gaussian HMM (test labels decoded causally), then name each cluster by its train mean correlation, highest = Crisis.
results = fit_and_rank_regimes(X_train, X_test, train["mu_values"])

for method, result in results.items():
    print(method, "train mean correlation by regime:", dict(zip(config.REGIME_NAMES, np.round(result["mean_mu"][result["order"]], 3))))

print("HMM expected dwell (windows):", dict(zip(config.REGIME_NAMES, np.round(results["hmm"]["dwell"], 1))))

In [ ]:
# K-means regime timeline over the mean correlation, train and test on one axis.
kmeans = results["kmeans"]
plot_regime_timeline(list(train["window_dates"]), train["mu_values"], kmeans["train_labels"], list(test["window_dates"]), test["mu_values"], kmeans["test_labels"], kmeans["rank_of_cluster"])
plt.show()

In [ ]:
# Validate each method's Crisis regime against VIX > 30 on every window's end date: precision, recall and F1, train and test separately, both from causally decoded labels.
vix = load_vix()
validation = {method: {split: validate_regime(result["train_causal_labels" if split == "train" else "test_labels"], part["window_dates"], vix, config.VIX_CRISIS_THRESHOLD, result["crisis_id"]) for split, part in (("train", train), ("test", test))} for method, result in results.items()}
pd.DataFrame({(method, split): dict(zip(["precision", "recall", "f1"], v["point"])) for method, splits in validation.items() for split, v in splits.items()}).round(3)

In [ ]:
# K-means Crisis windows shaded over the VIX.
plot_vix_overlay(pd.DatetimeIndex(train["window_dates"]), validation["kmeans"]["train"], pd.DatetimeIndex(test["window_dates"]), validation["kmeans"]["test"], WINDOW_LENGTH)
plt.show()

In [ ]:
# Empirical K-means transition matrices on train and test windows: persistence is the diagonal.
(train_transitions, train_occupancy), (test_transitions, test_occupancy) = (empirical_transitions(kmeans[f"{split}_labels"], config.N_REGIMES, kmeans["order"]) for split in ("train", "test"))
plot_transition_comparison(train_transitions, test_transitions, train_occupancy, test_occupancy, f"{INDEX_CODE} T={WINDOW_LENGTH} {GRAPH_TYPE} (kmeans)")
plt.show()